In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS googleads_silver
""")

DataFrame[]

In [0]:
bronze_tables = [
    table.name
    for table in spark.catalog.listTables("googleads_bronze")
]

print(bronze_tables)

['bronze_ad_group_performance', 'bronze_ad_performance', 'bronze_age_performance', 'bronze_budget_information', 'bronze_campaign_details', 'bronze_campaign_performance', 'bronze_conversion_actions', 'bronze_conversion_performance', 'bronze_customer_info', 'bronze_device_performance', 'bronze_gender_performance', 'bronze_geographic_performance', 'bronze_keyword_performance', 'bronze_landing_page_performance', 'bronze_search_terms']


In [0]:
campaign_df = spark.table("googleads_bronze.bronze_campaign_performance")

keyword_df = spark.table("googleads_bronze.bronze_keyword_performance")

search_term_df = spark.table("googleads_bronze.bronze_search_terms")

conversion_df = spark.table("googleads_bronze.bronze_conversion_performance")

device_df = spark.table("googleads_bronze.bronze_device_performance")

gender_df = spark.table("googleads_bronze.bronze_gender_performance")

age_df = spark.table("googleads_bronze.bronze_age_performance")

geo_df = spark.table("googleads_bronze.bronze_geographic_performance")

ad_df = spark.table("googleads_bronze.bronze_ad_performance")

ad_group_df = spark.table("googleads_bronze.bronze_ad_group_performance")

landing_page_df = spark.table("googleads_bronze.bronze_landing_page_performance")

budget_df = spark.table("googleads_bronze.bronze_budget_information")

campaign_details_df = spark.table("googleads_bronze.bronze_campaign_details")

conversion_action_df = spark.table("googleads_bronze.bronze_conversion_actions")

customer_info_df = spark.table("googleads_bronze.bronze_customer_info")

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    regexp_extract,
    regexp_replace,
    translate,
    lit,
    coalesce
)

In [0]:
from pyspark.sql.functions import col, when, to_date

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in campaign_df.columns]

# Step 2: Null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_ctr",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "campaign_status",
    "campaign_advertising_channel_type",
    "campaign_resource_name"
]

# Step 3: Transform
df_campaigns = (
    campaign_df
    .toDF(*new_column_names)

    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)

    # Convert date
    .withColumn(
        "segments_date",
        to_date(col("segments_date"), "yyyy-MM-dd")
    )

    # Cast numeric columns
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        "campaign_status",
        "campaign_advertising_channel_type",
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

df_campaigns.printSchema()
display(df_campaigns)

root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- campaign_status: string (nullable = false)
 |-- campaign_advertising_channel_type: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,campaign_status,campaign_advertising_channel_type,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-12-20,21771245709,Azure Data Engineering -Display - Image,ENABLED,DISPLAY,16908,1440,0.0,493.380251,8.516678495386799,0.34262517430555556,0.0,0.0
2025-12-21,21771245709,Azure Data Engineering -Display - Image,ENABLED,DISPLAY,1831,263,0.0,59.266752,14.363735663571818,0.22534886692015207,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,ENABLED,DISPLAY,22702,3060,0.0,993.128612,13.478988635362523,0.3245518339869281,0.0,0.0
2025-06-15,22489991406,Hyderabad,PAUSED,SEARCH,197,18,0.0,84.361274,9.137055837563452,4.686737444444444,0.0,0.0
2025-06-16,22489991406,Hyderabad,PAUSED,SEARCH,316,44,0.0,235.307321,13.924050632911392,5.347893659090909,0.0,0.0
2025-06-17,22489991406,Hyderabad,PAUSED,SEARCH,134,11,0.0,52.860925,8.208955223880597,4.805538636363637,0.0,0.0
2025-06-18,22489991406,Hyderabad,PAUSED,SEARCH,178,14,2.0,165.397077,7.865168539325842,11.814076928571428,14.285714285714286,82.6985385
2025-06-19,22489991406,Hyderabad,PAUSED,SEARCH,119,6,0.0,62.32,5.042016806722689,10.386666666666667,0.0,0.0
2025-06-20,22489991406,Hyderabad,PAUSED,SEARCH,865,39,0.0,133.73552,4.508670520231214,3.4291158974358975,0.0,0.0
2025-06-21,22489991406,Hyderabad,PAUSED,SEARCH,6668,251,0.0,1190.67938,3.764247150569886,4.743742549800797,0.0,0.0


In [0]:
from pyspark.sql.functions import col, when

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in keyword_df.columns]



# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "ad_group_id",
    "ad_group_name",
    "ad_group_criterion_keyword_text",
    "ad_group_criterion_keyword_match_type"
]

# Step 3: Transform
df_keywords = (
    keyword_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
         "segments_date",
        "campaign_id",
        "campaign_name",
        "ad_group_id",
        "ad_group_name",

        col("ad_group_criterion_keyword_text").alias("keyword"),
        col("ad_group_criterion_keyword_match_type").alias("keyword_match_type"),

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
df_keywords.printSchema()

display(df_keywords)

root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_id: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- keyword: string (nullable = false)
 |-- keyword_match_type: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,ad_group_id,ad_group_name,keyword,keyword_match_type,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,cloud database course,BROAD,114,2,0.0,31.0,1.7543859649122806,15.5,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,Data Engineer certification training,BROAD,16,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,snowpro advanced data engineer,BROAD,8,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,genai course,BROAD,50,2,0.0,149.62,4.0,74.81,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,Python for Data Engineering,BROAD,92,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,AWS Data Engineering course,BROAD,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,Snowflake certification course,BROAD,11,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,Generative AI course online,BROAD,84,3,0.0,62.42,3.5714285714285716,20.80666666666667,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,Snowflake Data Engineering course,BROAD,8,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-27,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,AWS Glue training,BROAD,6,0,0.0,0.0,0.0,0.0,0.0,0.0


In [0]:
from pyspark.sql.functions import col, when


# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in search_term_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "ad_group_id",
    "ad_group_name",
    "search_term_view_search_term"
]

# Step 3: Transform
df_search_terms = (
    search_term_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        "ad_group_id",
        "ad_group_name",
        col("search_term_view_search_term").alias("search_term"),

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
df_search_terms.printSchema()

display(df_search_terms)

root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_id: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- search_term: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,ad_group_id,ad_group_name,search_term,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,databricks data engineering associate,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,databricks data engineering associate certification,2,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,databricks for beginners,2,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,databricks for professionals,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,databricks full course in hindi,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,databricks learning path,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,databricks training,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,datacamp data engineering,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,dataflair data structure,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-24,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,dataflow gcp python,1,0,0.0,0.0,0.0,0.0,0.0,0.0


In [0]:
from pyspark.sql.functions import col,to_date, when


# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in conversion_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_conversions",
    "metrics_conversions_value"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "campaign_resource_name",
    "segments_conversion_action_name"
]

# Step 3: Transform
df_conversions = (
    conversion_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn(
        "metrics_conversions",
        col("metrics_conversions").cast("double")
    )

    .withColumn(
        "metrics_conversions_value",
        col("metrics_conversions_value").cast("double")
    )



    # Final columns
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        col("segments_conversion_action_name").alias("conversion_action_name"),

        "metrics_conversions",
        "metrics_conversions_value"


    )
)

# Verify
df_conversions.printSchema()

display(df_conversions)

root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- conversion_action_name: string (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- metrics_conversions_value: double (nullable = false)



segments_date,campaign_id,campaign_name,conversion_action_name,metrics_conversions,metrics_conversions_value
2025-06-18,22489991406,Hyderabad,Calls from ads,2.0,2.0
2025-12-23,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,2.0,2.0
2025-12-24,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,2.0,2.0
2025-12-26,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,3.0,3.0
2025-12-27,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,5.0,5.0
2025-12-28,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,6.0,6.0
2025-12-29,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,4.0,4.0
2025-12-30,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,4.0,4.0
2025-12-31,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,2.0,2.0
2026-01-01,23393627674,Azure-Data-Engineering-Jan-2026,Lead form - Submit,4.0,4.0


In [0]:
from pyspark.sql.functions import col, when


# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in device_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "campaign_resource_name",
    "segments_device"
]

# Step 3: Transform
df_devices = (
    device_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
        "segments_date",
        
        "campaign_id",
        "campaign_name",
        col("segments_device").alias("device"),

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
df_devices.printSchema()

display(df_devices)

root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- device: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,device,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-12-20,21771245709,Azure Data Engineering -Display - Image,DESKTOP,1,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-12-20,21771245709,Azure Data Engineering -Display - Image,MOBILE,16689,1415,0.0,484.548865,8.478638624243514,0.3424373604240282,0.0,0.0
2025-12-20,21771245709,Azure Data Engineering -Display - Image,TABLET,218,25,0.0,8.831386,11.46788990825688,0.35325544000000003,0.0,0.0
2025-12-21,21771245709,Azure Data Engineering -Display - Image,MOBILE,1785,254,0.0,57.07897,14.2296918767507,0.22472035433070867,0.0,0.0
2025-12-21,21771245709,Azure Data Engineering -Display - Image,TABLET,46,9,0.0,2.187782,19.565217391304348,0.24308688888888888,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,OTHER,5,1,0.0,0.698256,20.0,0.698256,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,MOBILE,22544,3044,0.0,986.478377,13.502484031227821,0.32407305420499344,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,TABLET,153,15,0.0,5.951979,9.803921568627452,0.3967986,0.0,0.0
2025-06-15,22489991406,Hyderabad,DESKTOP,3,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-15,22489991406,Hyderabad,MOBILE,193,18,0.0,84.361274,9.32642487046632,4.686737444444444,0.0,0.0


In [0]:
from pyspark.sql.functions import col, when

# =====================================================
# Gender Performance → Silver Gender
# =====================================================

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in gender_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "ad_group_criterion_gender_type_",
    "ad_group_criterion_resource_name",
    "gender_view_resource_name"
]

# Step 3: Transform
df_gender = (
    gender_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
        "segments_date",
        col("ad_group_criterion_gender_type_").alias("gender"),

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
df_gender.printSchema()

display(df_gender)

root
 |-- segments_date: date (nullable = true)
 |-- gender: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,gender,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-12-20,MALE,10572,882,0.0,336.372677,8.342792281498298,0.3813749172335601,0.0,0.0
2025-12-20,FEMALE,2583,214,0.0,73.694564,8.284939992257065,0.3443671214953271,0.0,0.0
2025-12-20,UNDETERMINED,3753,344,0.0,83.31301,9.166000532907008,0.24218898255813956,0.0,0.0
2025-12-21,MALE,1132,180,0.0,39.08224,15.901060070671377,0.21712355555555554,0.0,0.0
2025-12-21,FEMALE,411,51,0.0,12.698735,12.408759124087592,0.2489948039215686,0.0,0.0
2025-12-21,UNDETERMINED,288,32,0.0,7.485777,11.11111111111111,0.23393053125,0.0,0.0
2025-12-22,MALE,11609,1610,0.0,554.651903,13.868550262727194,0.34450428757763973,0.0,0.0
2025-12-22,FEMALE,3208,427,0.0,136.02177,13.310473815461346,0.31855215456674474,0.0,0.0
2025-12-22,UNDETERMINED,7885,1023,0.0,302.454939,12.974001268230818,0.2956548768328446,0.0,0.0
2025-06-15,MALE,21,1,0.0,14.73,4.761904761904762,14.73,0.0,0.0


In [0]:
from pyspark.sql.functions import col, when



# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in age_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "ad_group_criterion_age_range_type_",
    "ad_group_criterion_resource_name",
    "age_range_view_resource_name"
]

# Step 3: Transform
df_age = (
    age_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
        "segments_date",
        col("ad_group_criterion_age_range_type_").alias("age_range"),

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
df_age.printSchema()

display(df_age)

root
 |-- segments_date: date (nullable = true)
 |-- age_range: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,age_range,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-12-20,AGE_RANGE_18_24,3459,285,0.0,117.424114,8.239375542064181,0.4120144350877193,0.0,0.0
2025-12-20,AGE_RANGE_25_34,4021,322,0.0,140.14654,8.00795821934842,0.435237701863354,0.0,0.0
2025-12-20,AGE_RANGE_35_44,2805,210,0.0,76.674225,7.4866310160427805,0.3651153571428572,0.0,0.0
2025-12-20,AGE_RANGE_45_54,1230,118,0.0,32.593968,9.59349593495935,0.2762200677966101,0.0,0.0
2025-12-20,AGE_RANGE_55_64,753,74,0.0,16.767238,9.827357237715804,0.2265842972972973,0.0,0.0
2025-12-20,AGE_RANGE_65_UP,516,47,0.0,16.831235,9.108527131782946,0.3581113829787234,0.0,0.0
2025-12-20,AGE_RANGE_UNDETERMINED,4124,384,0.0,92.942931,9.311348205625606,0.2420388828125,0.0,0.0
2025-12-21,AGE_RANGE_18_24,343,56,0.0,13.161333,16.3265306122449,0.23502380357142857,0.0,0.0
2025-12-21,AGE_RANGE_25_34,475,65,0.0,14.675505,13.68421052631579,0.22577699999999998,0.0,0.0
2025-12-21,AGE_RANGE_35_44,327,56,0.0,11.889223,17.125382262996943,0.21230755357142855,0.0,0.0


In [0]:
from pyspark.sql.functions import col, when


# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in geo_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "campaign_resource_name",
    "geographic_view_country_criterion_id",
    "geographic_view_resource_name"
]

# Step 3: Transform
df_geographic = (
    geo_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        col("geographic_view_country_criterion_id").alias("country_criterion_id"),

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
print("----- Final Schema -----")
df_geographic.printSchema()

display(df_geographic)

----- Final Schema -----
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- country_criterion_id: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,country_criterion_id,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-12-20,21771245709,Azure Data Engineering -Display - Image,2356,7989,816,0.0,256.826898,10.214044310927525,0.31473884558823534,0.0,0.0
2025-12-20,21771245709,Azure Data Engineering -Display - Image,2356,8919,624,0.0,236.553353,6.996300033636058,0.37909191185897434,0.0,0.0
2025-12-21,21771245709,Azure Data Engineering -Display - Image,2356,1113,172,0.0,39.133539,15.45372866127583,0.22752057558139535,0.0,0.0
2025-12-21,21771245709,Azure Data Engineering -Display - Image,2356,718,91,0.0,20.133213,12.674094707520892,0.22124409890109892,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,2356,14032,1922,0.0,639.989716,13.697263397947548,0.332981121748179,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,2356,8670,1138,0.0,353.138896,13.125720876585929,0.310315374340949,0.0,0.0
2025-06-15,22489991406,Hyderabad,2356,73,3,0.0,12.455625,4.109589041095891,4.1518749999999995,0.0,0.0
2025-06-15,22489991406,Hyderabad,2356,124,15,0.0,71.905649,12.096774193548388,4.793709933333333,0.0,0.0
2025-06-16,22489991406,Hyderabad,2356,162,15,0.0,58.77457,9.25925925925926,3.9183046666666663,0.0,0.0
2025-06-16,22489991406,Hyderabad,2356,154,29,0.0,176.532751,18.83116883116883,6.08733624137931,0.0,0.0


In [0]:
from pyspark.sql.functions import col, when


# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in ad_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "ad_group_id",
    "ad_group_name",
    "ad_group_ad_ad_id"
]

# Step 3: Transform
df_ads = (
    ad_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",

        "ad_group_id",
        "ad_group_name",

        col("ad_group_ad_ad_id").alias("ad_id"),

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
print("----- Final Schema -----")
df_ads.printSchema()

display(df_ads)

----- Final Schema -----
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_id: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- ad_id: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,ad_group_id,ad_group_name,ad_id,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-12-20,21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,715897121402,16908,1440,0.0,493.380251,8.516678495386799,0.34262517430555556,0.0,0.0
2025-12-21,21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,715897121402,1831,263,0.0,59.266752,14.363735663571818,0.22534886692015207,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,715897121402,22702,3060,0.0,993.128612,13.478988635362523,0.3245518339869281,0.0,0.0
2025-06-15,22489991406,Hyderabad,177495674774,Ad group 1,749552696579,197,18,0.0,84.361274,9.137055837563452,4.686737444444444,0.0,0.0
2025-06-16,22489991406,Hyderabad,177495674774,Ad group 1,749552696579,316,44,0.0,235.307321,13.924050632911392,5.347893659090909,0.0,0.0
2025-06-17,22489991406,Hyderabad,177495674774,Ad group 1,749552696579,134,11,0.0,52.860925,8.208955223880597,4.805538636363637,0.0,0.0
2025-06-18,22489991406,Hyderabad,177495674774,Ad group 1,749552696579,178,14,2.0,165.397077,7.865168539325842,11.814076928571428,14.285714285714286,82.6985385
2025-06-19,22489991406,Hyderabad,177495674774,Ad group 1,749552696579,119,6,0.0,62.32,5.042016806722689,10.386666666666667,0.0,0.0
2025-06-20,22489991406,Hyderabad,177495674774,Ad group 1,749552696579,865,39,0.0,133.73552,4.508670520231214,3.4291158974358975,0.0,0.0
2025-06-21,22489991406,Hyderabad,177495674774,Ad group 1,749552696579,6668,251,0.0,1190.67938,3.764247150569886,4.743742549800797,0.0,0.0


In [0]:

from pyspark.sql.functions import col, when

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in ad_group_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "ad_group_id",
    "ad_group_name"
]

# Step 3: Transform
df_ad_groups = (
    ad_group_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    # Cast datatypes
    .withColumn("campaign_id", col("campaign_id").cast("string"))
    .withColumn("ad_group_id", col("ad_group_id").cast("string"))

    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    # Cost
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    # CTR
    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    # CPC
    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    # CVR
    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    # CPA
    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    # Final columns
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        "ad_group_id",
        "ad_group_name",

        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",

        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

# Verify
df_ad_groups.printSchema()

display(df_ad_groups)

root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_id: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,ad_group_id,ad_group_name,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-12-20,21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,16908,1440,0.0,493.380251,8.516678495386799,0.34262517430555556,0.0,0.0
2025-12-21,21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,1831,263,0.0,59.266752,14.363735663571818,0.22534886692015207,0.0,0.0
2025-12-22,21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,22702,3060,0.0,993.128612,13.478988635362523,0.3245518339869281,0.0,0.0
2025-06-15,22489991406,Hyderabad,177495674774,Ad group 1,197,18,0.0,84.361274,9.137055837563452,4.686737444444444,0.0,0.0
2025-06-16,22489991406,Hyderabad,177495674774,Ad group 1,316,44,0.0,235.307321,13.924050632911392,5.347893659090909,0.0,0.0
2025-06-17,22489991406,Hyderabad,177495674774,Ad group 1,134,11,0.0,52.860925,8.208955223880597,4.805538636363637,0.0,0.0
2025-06-18,22489991406,Hyderabad,177495674774,Ad group 1,178,14,2.0,165.397077,7.865168539325842,11.814076928571428,14.285714285714286,82.6985385
2025-06-19,22489991406,Hyderabad,177495674774,Ad group 1,119,6,0.0,62.32,5.042016806722689,10.386666666666667,0.0,0.0
2025-06-20,22489991406,Hyderabad,177495674774,Ad group 1,865,39,0.0,133.73552,4.508670520231214,3.4291158974358975,0.0,0.0
2025-06-21,22489991406,Hyderabad,177495674774,Ad group 1,6668,251,0.0,1190.67938,3.764247150569886,4.743742549800797,0.0,0.0


In [0]:
from pyspark.sql.functions import col, when

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in landing_page_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "metrics_clicks",
    "metrics_conversions",
    "metrics_cost_micros",
    "metrics_impressions"
]

string_cols_to_fill = [
    "campaign_id",
    "campaign_name",
    "landing_page_view_unexpanded_final_url"
]

# Step 3: Transform
df_landing_pages = (
    landing_page_df
    .toDF(*new_column_names)

    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)
    .withColumn(
    "segments_date",
    to_date(col("segments_date"), "yyyy-MM-dd")
)

    .withColumn("campaign_id", col("campaign_id").cast("string"))
    .withColumn("metrics_clicks", col("metrics_clicks").cast("long"))
    .withColumn("metrics_impressions", col("metrics_impressions").cast("long"))
    .withColumn("metrics_cost_micros", col("metrics_cost_micros").cast("double"))
    .withColumn("metrics_conversions", col("metrics_conversions").cast("double"))

    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    )

    .withColumn(
        "ctr_pct",
        when(
            col("metrics_impressions") > 0,
            col("metrics_clicks") * 100.0 / col("metrics_impressions")
        ).otherwise(0)
    )

    .withColumn(
        "cpc_inr",
        when(
            col("metrics_clicks") > 0,
            col("cost_inr") / col("metrics_clicks")
        ).otherwise(0)
    )

    .withColumn(
        "cvr_pct",
        when(
            col("metrics_clicks") > 0,
            col("metrics_conversions") * 100.0 / col("metrics_clicks")
        ).otherwise(0)
    )

    .withColumn(
        "cpa_inr",
        when(
            col("metrics_conversions") > 0,
            col("cost_inr") / col("metrics_conversions")
        ).otherwise(0)
    )

    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        col("landing_page_view_unexpanded_final_url").alias("landing_page_url"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "cost_inr",
        "ctr_pct",
        "cpc_inr",
        "cvr_pct",
        "cpa_inr"
    )
)

df_landing_pages.printSchema()
display(df_landing_pages)

root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- landing_page_url: string (nullable = false)
 |-- metrics_impressions: long (nullable = true)
 |-- metrics_clicks: long (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cpa_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,landing_page_url,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,ctr_pct,cpc_inr,cvr_pct,cpa_inr
2025-06-15,22489991406,Hyderabad,https://academyofdata.ai/data-science/,40,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-16,22489991406,Hyderabad,https://academyofdata.ai/data-science/,66,3,0.0,58.700666,4.545454545454546,19.566888666666667,0.0,0.0
2025-06-17,22489991406,Hyderabad,https://academyofdata.ai/data-science/,14,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-18,22489991406,Hyderabad,https://academyofdata.ai/data-science/,17,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-19,22489991406,Hyderabad,https://academyofdata.ai/data-science/,9,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-20,22489991406,Hyderabad,https://academyofdata.ai/data-science/,3,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-21,22489991406,Hyderabad,https://academyofdata.ai/data-science/,146,0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-22,22489991406,Hyderabad,https://academyofdata.ai/data-science/,181,1,0.0,7.0,0.5524861878453039,7.0,0.0,0.0
2025-06-23,22489991406,Hyderabad,https://academyofdata.ai/data-science/,192,2,0.0,11.812065,1.0416666666666667,5.9060325,0.0,0.0
2025-06-24,22489991406,Hyderabad,https://academyofdata.ai/data-science/,56,0,0.0,0.0,0.0,0.0,0.0,0.0


In [0]:

from pyspark.sql.functions import col

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in budget_df.columns]

# Step 2: Define columns for null handling
numeric_cols_to_fill = [
    "campaign_budget_amount_micros"
]

string_cols_to_fill = [
    "campaign_budget_id",
    "campaign_budget_name",
    "campaign_budget_resource_name"
]

# Step 3: Transform
df_budget = (
    budget_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna(0, subset=numeric_cols_to_fill)
    .fillna("Unknown", subset=string_cols_to_fill)

    # Cast datatypes
    .withColumn(
        "campaign_budget_amount_micros",
        col("campaign_budget_amount_micros").cast("double")
    )

    .withColumn(
        "campaign_budget_id",
        col("campaign_budget_id").cast("string")
    )

    # Convert micros to INR
    .withColumn(
        "budget_inr",
        col("campaign_budget_amount_micros") / 1000000
    )

    # Final columns
    .select(
        "campaign_budget_id",
        "campaign_budget_name",
        "budget_inr"
    )
)

# Verify
df_budget.printSchema()

display(df_budget)

root
 |-- campaign_budget_id: string (nullable = false)
 |-- campaign_budget_name: string (nullable = false)
 |-- budget_inr: double (nullable = true)



campaign_budget_id,campaign_budget_name,budget_inr
13519012233,Excel in Data Engineering_obsolete,341.0
13779178023,Excel in Data Engineering,341.0
13969318520,Azure Data Engineering -Display - Image,500.0
13975409178,Azure Data Engineering -Display - text,500.0
14348313107,Leads-Search-24Feb2025,1000.0
14516931095,Hyderabad,1500.0
14898232496,Data Engineering (AWS-Sept),1003.5
14928117082,10-Sept AWS Snowflake,1050.0
15234300935,Azure-Data-Engineering-Jan-2026,500.0
15516948212,april 21st,500.0


In [0]:
from pyspark.sql.functions import col

# Rename columns
new_column_names = [
    c.replace(".", "_")
    for c in campaign_details_df.columns
]

# Transform campaign details
df_campaign_details = (
    campaign_details_df
    .toDF(*new_column_names)

    .fillna(
        "Unknown",
        subset=[
            "campaign_id",
            "campaign_name",
            "campaign_status",
            "campaign_serving_status",
            "campaign_advertising_channel_type",
            "campaign_bidding_strategy_type",
            "campaign_budget_id"
        ]
    )

    .withColumn(
        "campaign_id",
        col("campaign_id").cast("string")
    )

    .withColumn(
        "campaign_budget_id",
        col("campaign_budget_id").cast("string")
    )

    .select(
        "campaign_id",
        "campaign_name",
        "campaign_status",
        "campaign_serving_status",
        "campaign_advertising_channel_type",
        "campaign_bidding_strategy_type",
        "campaign_budget_id"
    )
)

# Verify
df_campaign_details.printSchema()
display(df_campaign_details)

# Save to Silver
df_campaign_details.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("googleads_silver.silver_campaign_details")

root
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- campaign_status: string (nullable = false)
 |-- campaign_serving_status: string (nullable = false)
 |-- campaign_advertising_channel_type: string (nullable = false)
 |-- campaign_bidding_strategy_type: string (nullable = false)
 |-- campaign_budget_id: string (nullable = false)



campaign_id,campaign_name,campaign_status,campaign_serving_status,campaign_advertising_channel_type,campaign_bidding_strategy_type,campaign_budget_id
21519858184,Excel in Data Engineering,PAUSED,SERVING,PERFORMANCE_MAX,MAXIMIZE_CONVERSIONS,13779178023
21771245709,Azure Data Engineering -Display - Image,ENABLED,ENDED,DISPLAY,MAXIMIZE_CONVERSIONS,13969318520
21781297084,Azure Data Engineering -Display - text,PAUSED,ENDED,SEARCH,MAXIMIZE_CONVERSIONS,13975409178
22271705916,Leads-Search-24Feb2025,PAUSED,ENDED,SEARCH,MAXIMIZE_CONVERSIONS,14348313107
22489991406,Hyderabad,PAUSED,ENDED,SEARCH,MAXIMIZE_CONVERSIONS,14516931095
22973632076,Data Engineering (AWS-Sept),PAUSED,SERVING,PERFORMANCE_MAX,MAXIMIZE_CONVERSIONS,14898232496
22998241842,10-Sept AWS Snowflake,PAUSED,ENDED,SEARCH,MAXIMIZE_CONVERSION_VALUE,14928117082
23393627674,Azure-Data-Engineering-Jan-2026,PAUSED,SERVING,SEARCH,MAXIMIZE_CONVERSIONS,15234300935
23770266216,april 21st,ENABLED,SERVING,PERFORMANCE_MAX,MAXIMIZE_CONVERSIONS,15516948212
23770277166,APRIL 21ST,ENABLED,SERVING,SEARCH,MAXIMIZE_CONVERSIONS,15521655015


In [0]:
campaign_details_df = spark.table("googleads_silver.silver_campaign_details")

In [0]:
from pyspark.sql.functions import regexp_extract

df_campaign_details = (
    campaign_details_df
    .toDF(*new_column_names)

    .withColumn(
        "campaign_budget_id",
        regexp_extract(
            col("campaign_campaign_budget"),
            r'campaignBudgets/(\\d+)',
            1
        )
    )

    .select(
        "campaign_id",
        "campaign_name",
        "campaign_status",
        "campaign_serving_status",
        "campaign_advertising_channel_type",
        "campaign_bidding_strategy_type",
        "campaign_budget_id"
    )
)

In [0]:
from pyspark.sql.functions import col

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in conversion_action_df.columns]

# Step 2: Define columns for null handling
string_cols_to_fill = [
    "conversion_action_id",
    "conversion_action_name",
    "conversion_action_resource_name",
    "conversion_action_status",
    "conversion_action_type_"
]

# Step 3: Transform
df_conversion_action = (
    conversion_action_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna("Unknown", subset=string_cols_to_fill)

    # Cast datatype
    .withColumn(
        "conversion_action_id",
        col("conversion_action_id").cast("string")
    )

    # Final columns
    .select(
        "conversion_action_id",
        "conversion_action_name",
        "conversion_action_type_",
        "conversion_action_status"
    )
)

# Verify
df_conversion_action.printSchema()

display(df_conversion_action)

root
 |-- conversion_action_id: string (nullable = false)
 |-- conversion_action_name: string (nullable = false)
 |-- conversion_action_type_: string (nullable = false)
 |-- conversion_action_status: string (nullable = false)



conversion_action_id,conversion_action_name,conversion_action_type_,conversion_action_status
6858948899,academyofdata.in (web) purchase,GOOGLE_ANALYTICS_4_PURCHASE,HIDDEN
6859177800,Calls from ads,AD_CALL,ENABLED
6859177803,Clicks to call,GOOGLE_HOSTED,ENABLED
6859183569,Book appointment,GOOGLE_ANALYTICS_4_CUSTOM,ENABLED
6864326819,Android installs (all other apps),ANDROID_INSTALLS_ALL_OTHER_APPS,ENABLED
7052820665,Contact (Form submission https://academyofdata.ai/contact-us/contact-us/),WEBPAGE_CODELESS,ENABLED
7114762484,Contact,WEBPAGE,ENABLED
7117812172,Local actions - Directions,GOOGLE_HOSTED,ENABLED
7122187177,Local actions - Other engagements,GOOGLE_HOSTED,ENABLED
7123172631,Local actions - Website visits,GOOGLE_HOSTED,ENABLED


In [0]:


from pyspark.sql.functions import col

# Step 1: Rename columns
new_column_names = [c.replace(".", "_") for c in customer_info_df.columns]

# Step 2: Define columns for null handling
string_cols_to_fill = [
    "customer_id",
    "customer_currency_code",
    "customer_resource_name",
    "customer_time_zone"
]

# Step 3: Transform
df_customer_info = (
    customer_info_df
    .toDF(*new_column_names)

    # Handle nulls
    .fillna("Unknown", subset=string_cols_to_fill)

    # Cast datatypes
    .withColumn(
        "customer_id",
        col("customer_id").cast("string")
    )

    # Final columns
    .select(
        "customer_id",
        "customer_currency_code",
        "customer_time_zone"
    )
)

# Verify
df_customer_info.printSchema()

display(df_customer_info)


root
 |-- customer_id: string (nullable = false)
 |-- customer_currency_code: string (nullable = false)
 |-- customer_time_zone: string (nullable = false)



customer_id,customer_currency_code,customer_time_zone
1401815809,INR,Asia/Calcutta


In [0]:
df_campaigns.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_campaign_performance"
)

df_keywords.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_keyword_performance"
)

df_search_terms.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_search_terms"
)

df_conversions.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_conversion_performance"
)

df_devices.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_device_performance"
)

df_gender.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_gender_performance"
)

df_age.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_age_performance"
)

df_geographic.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_geographic_performance"
)

df_ads.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_ad_performance"
)

df_ad_groups.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_ad_group_performance"
)

df_landing_pages.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_landing_page_performance"
)

df_budget.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_budget_information"
)

df_campaign_details.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable("googleads_silver.silver_campaign_details")

df_conversion_action.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_conversion_actions"
)

df_customer_info.write.mode("overwrite").format("delta").saveAsTable(
    "googleads_silver.silver_customer_info"
)

In [0]:
spark.sql("SHOW TABLES IN googleads_silver").show()

+----------------+--------------------+-----------+
|        database|           tableName|isTemporary|
+----------------+--------------------+-----------+
|googleads_silver|silver_ad_group_p...|      false|
|googleads_silver|silver_ad_perform...|      false|
|googleads_silver|silver_age_perfor...|      false|
|googleads_silver|silver_budget_inf...|      false|
|googleads_silver|silver_campaign_d...|      false|
|googleads_silver|silver_campaign_p...|      false|
|googleads_silver|silver_conversion...|      false|
|googleads_silver|silver_conversion...|      false|
|googleads_silver|silver_customer_info|      false|
|googleads_silver|silver_device_per...|      false|
|googleads_silver|silver_gender_per...|      false|
|googleads_silver|silver_geographic...|      false|
|googleads_silver|silver_keyword_pe...|      false|
|googleads_silver|silver_landing_pa...|      false|
|googleads_silver| silver_search_terms|      false|
+----------------+--------------------+-----------+

